# 098 — Procedencia, marcas y autenticidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**C2PA**: manifiesto adjunto al activo con **aserciones** (qué herramienta/modelo
intervino), un **reclamo** con el *hard binding* (hash criptográfico de los bytes del
activo) y la **firma** del reclamo con un certificado X.509. Cada edición añade un
manifiesto que referencia el anterior → **cadena de procedencia**. Verificar =
recalcular hash + validar firmas. Acredita *quién firmó qué proceso*, no que el
contenido sea "verdadero"; y la ausencia de manifiesto no prueba nada.

**Marca de agua para texto** (Kirchenbauer et al., 2023): antes de emitir cada token,
un hash del token anterior parte el vocabulario en lista **verde** (fracción γ) y
**roja**; se suma un sesgo δ al logit de los verdes. El texto marcado tiene un exceso
estadístico de tokens verdes, invisible a la lectura. **Detección** = test de
hipótesis: bajo H₀ el conteo verde k ~ Binomial(T, γ) y
`z = (k − γT) / √(Tγ(1−γ))`. **SynthID-Text** (Nature 2024) es la versión desplegada
en producción de esta familia de ideas.

**Detección pasiva** (clasificar contenido sin cooperación del generador) es frágil y
con falsos positivos sesgados; la procedencia activa y las marcas requieren
cooperación pero dan garantías verificables. Límites comunes: recorte y recompresión
eliminan metadatos, la paráfrasis diluye la señal verde y el **lavado** (regenerar con
otro modelo) elimina la marca.

## 🧮 Ejemplo de referencia

Marca greenlist con γ = 0.5, T = 100 tokens, k = 70 verdes observados:
esperado γT = 50, σ = √(100·0.25) = 5 → **z = (70 − 50)/5 = 4.0**
(p unilateral ≈ 3.2 × 10⁻⁵: marca presente). Con k = 55, z = 1.0: compatible con el
azar. La detección es un continuo de confianza, no un sello binario. Verifica el
cálculo a mano antes de ejecutar el laboratorio.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("safety", seed=98)
show(result)


## Reflexión

1. Con γ = 0.5, un 70 % de tokens verdes da z = 4.0 si T = 100, pero solo z ≈ 1.26 si T = 10. ¿Por qué la longitud del texto es decisiva para el detector y qué implica para fragmentos cortos (titulares, tuits)?
2. Una imagen generada por IA puede llevar un manifiesto C2PA perfectamente válido. ¿Qué acredita exactamente esa firma y por qué la ausencia de manifiesto no prueba origen humano?
3. La paráfrasis o el lavado con otro modelo eliminan la marca de agua sin conocer la clave. ¿Qué ataque análogo sufre C2PA (eliminar el manifiesto) y por qué la garantía de ambos esquemas es unidireccional?